In [ ]:
"""
MATSim population editor with probabilistic conversion to micro_car:

- Eligibility: persons with subpopulation == "person", household_size <= 2, and total distance of <leg mode="car"> in the SELECTED plan below DIST_THRESHOLD_KM.
- Randomly sample a share (CONVERT_SHARE) of eligibles.
- For sampled persons:
    * Change all 'car' legs to 'micro_car' in the SELECTED plan only.
    * Set subpopulation to "potMCUser" only if ≥1 car leg was changed.
- Robust I/O for .xml and .xml.gz; preserves MATSim DTD.
- Reports detailed statistics.
"""

from lxml import etree
from pathlib import Path
from collections import Counter
import gzip
import random

# --------------------- CONFIG ---------------------
IN_PATH  = Path("/home/shahriar-iqbal-zame/IdeaProjects/matsim-berlin/input/v6.4/berlin-v6.4-10pct.plans.xml.gz")
OUT_PATH = Path("/home/shahriar-iqbal-zame/IdeaProjects/matsim-berlin/input/v6.4/MC50pct_berlin-v6.4-10pct.plans.xml.gz")

# Eligibility parameters
DIST_THRESHOLD_KM = 90.0           # threshold on total distance of car legs in SELECTED plan

# Conversion parameters
MICRO_MODE   = "micro_car"         # new mode to assign
CAR_MODE     = "car"
CONVERT_SHARE = 0.5                # e.g., 0.5 => convert 50% of eligible persons
RANDOM_SEED   = 42                 # set to None for non-deterministic; otherwise reproducible sampling

# MATSim DTD (kept identical to the input’s)
DTD_POPULATION = '<!DOCTYPE population SYSTEM "http://www.matsim.org/files/dtd/population_v6.dtd">'

# --------------------- HELPERS ---------------------
def parse_xml_maybe_gz(path: Path, parser: etree.XMLParser) -> etree._ElementTree:
    """Parse XML from .xml or .xml.gz path using the provided parser."""
    if path.suffix == ".gz":
        with gzip.open(path, "rb") as fh:
            return etree.parse(fh, parser)
    return etree.parse(str(path), parser)

def write_xml_maybe_gz(path: Path, xml_bytes: bytes) -> None:
    """Write XML bytes to .xml or .xml.gz path (gzip-compress if .gz)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix == ".gz":
        with gzip.open(path, "wb") as fh:
            fh.write(xml_bytes)
    else:
        path.write_bytes(xml_bytes)

def get_person_attr(person_el, name, default=None, cast=str):
    """Read a person-level <attribute name=...> value; returns default on missing/empty/parse error."""
    attrs_el = person_el.find("attributes")
    if attrs_el is None:
        return default
    for a in attrs_el.findall("attribute"):
        if a.get("name") == name:
            txt = (a.text or "").strip()
            if txt == "":
                return default
            try:
                return cast(txt)
            except Exception:
                return default
    return default

def set_person_attr(person_el, name, value, java_class="java.lang.String"):
    """Set or create a person-level <attribute> with given name/class/value."""
    attrs_el = person_el.find("attributes")
    if attrs_el is None:
        attrs_el = etree.SubElement(person_el, "attributes")
    for a in attrs_el.findall("attribute"):
        if a.get("name") == name:
            a.set("class", java_class)
            a.text = str(value)
            return
    a = etree.SubElement(attrs_el, "attribute")
    a.set("name", name)
    a.set("class", java_class)
    a.text = str(value)

def get_selected_plan(person_el):
    """Return the selected plan element; if none marked selected='yes', return the first plan or None."""
    plans = person_el.findall("plan")
    if not plans:
        return None
    for p in plans:
        if p.get("selected", "").lower() == "yes":
            return p
    return plans[0]

def subpopulation_counts(root):
    """Return (total_persons, Counter of subpopulation attribute values) across all <person>."""
    cnt = Counter()
    total = 0
    for person in root.findall("person"):
        total += 1
        sp = get_person_attr(person, "subpopulation", default="(missing)", cast=str)
        cnt[sp] += 1
    return total, cnt

def compute_car_distance_km_and_count(plan_el):
    """
    Returns (total_km_of_car_legs, car_leg_count) for the SELECTED plan.
    Distance source: <leg mode="car"><route distance="..."> (meters) → km.
    No fallback to activity 'orig_dist'; only the route's distance is used.
    """
    total_km = 0.0
    car_leg_count = 0
    for el in list(plan_el):
        if el.tag != "leg":
            continue
        if (el.get("mode") or "").strip() != CAR_MODE:
            continue
        car_leg_count += 1
        route_el = el.find("route")
        if route_el is None:
            continue  # no distance available; skip
        dist_attr = route_el.get("distance")
        if dist_attr is None:
            continue  # no distance attribute; skip
        try:
            total_km += float(dist_attr) / 1000.0
        except Exception:
            # ignore unparsable distance; treat as zero
            pass
    return total_km, car_leg_count

def change_car_legs_to_micro_in_selected_plan(person_el, micro_mode: str) -> int:
    """
    Change all legs with mode='car' to mode=micro_mode in the SELECTED plan only.
    Also change their 'routingMode' attribute (if exists) from 'car' to micro_mode.
    Returns number of legs changed for this person.
    """
    plan = get_selected_plan(person_el)
    if plan is None:
        return 0
    changed = 0
    for leg in plan.findall("leg"):
        mode = (leg.get("mode") or "").strip()
        if mode == CAR_MODE:
            leg.set("mode", micro_mode)

            # --- ADD THIS BLOCK ---
            attrs_el = leg.find("attributes")
            if attrs_el is not None:
                for attr in attrs_el.findall("attribute"):
                    if attr.get("name") == "routingMode" and (attr.text or "").strip() == CAR_MODE:
                        attr.text = micro_mode
            # --- END ADDITION ---

                        # --- ADD THIS NEW BLOCK (access/egress + interaction updates) ---
            # Work on siblings around the changed leg: walk access/egress and "car interaction" activities.
            plan_children = list(plan)
            idx = plan_children.index(leg)

            def _set_walk_routing_mode(leg_el):
                if leg_el is not None and leg_el.tag == "leg" and (leg_el.get("mode") or "").strip() == "walk":
                    attrs = leg_el.find("attributes")
                    if attrs is not None:
                        for attr in attrs.findall("attribute"):
                            if attr.get("name") == "routingMode" and (attr.text or "").strip() == CAR_MODE:
                                attr.text = micro_mode

            def _upgrade_interaction(activity_el):
                if activity_el is not None and activity_el.tag == "activity":
                    if (activity_el.get("type") or "").strip() == "car interaction":
                        activity_el.set("type", f"{micro_mode} interaction")

            # immediate neighbors
            prev1 = plan_children[idx - 1] if idx - 1 >= 0 else None
            next1 = plan_children[idx + 1] if idx + 1 < len(plan_children) else None

            # if the immediate neighbor is an activity, the walk leg is usually two steps away
            prev2 = plan_children[idx - 2] if idx - 2 >= 0 else None
            next2 = plan_children[idx + 2] if idx + 2 < len(plan_children) else None

            # Update activities adjacent to the micro_car leg
            _upgrade_interaction(prev1)
            _upgrade_interaction(next1)

            # Update access/egress walk legs' routingMode
            # Typical pattern: walk, activity, micro_car, activity, walk
            # so check prev2/next2 when prev1/next1 are activities
            if prev1 is not None and prev1.tag == "leg":
                _set_walk_routing_mode(prev1)
            elif prev2 is not None and prev1 is not None and prev1.tag == "activity":
                _set_walk_routing_mode(prev2)

            if next1 is not None and next1.tag == "leg":
                _set_walk_routing_mode(next1)
            elif next2 is not None and next1 is not None and next1.tag == "activity":
                _set_walk_routing_mode(next2)
            # --- END NEW BLOCK ---


            changed += 1
    return changed

def pct(n, d):
    return (100.0 * n / d) if d else 0.0

# --------------------- MAIN ---------------------
parser = etree.XMLParser(remove_blank_text=False)
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)

tree = parse_xml_maybe_gz(IN_PATH, parser)
root = tree.getroot()

# --- Baseline counts before any edits
total_persons_before, subpops_before = subpopulation_counts(root)
orig_person_count = subpops_before.get("person", 0)

# --- First pass: identify eligibility under the specified criteria
eligible_ids = []              # person ids eligible for sampling
eligibility_info = {}          # person id -> (total_km, hh_size, car_leg_cnt) for optional diagnostics

for person in root.findall("person"):
    subpop = get_person_attr(person, "subpopulation", default=None, cast=str)
    if subpop != "person":
        continue

    hh_size = get_person_attr(person, "household_size", default=None, cast=int)
    if hh_size is None:
        continue

    plan = get_selected_plan(person)
    if plan is None:
        continue

    total_km, car_leg_cnt = compute_car_distance_km_and_count(plan)

    # Require at least one car leg and distance below threshold
    if (hh_size <= 2) and (car_leg_cnt > 0) and (total_km < DIST_THRESHOLD_KM):
        pid = person.get("id")
        eligible_ids.append(pid)
        eligibility_info[pid] = (total_km, hh_size, car_leg_cnt)

eligible_scanned = len(eligible_ids)

# --- Randomly sample a share of eligibles for conversion
sample_size = int(round(CONVERT_SHARE * eligible_scanned))
sample_size = max(0, min(sample_size, eligible_scanned))  # clamp to [0, eligible_scanned]
selected_ids = set(random.sample(eligible_ids, sample_size)) if sample_size > 0 else set()

# --- Second pass: apply conversions ONLY to sampled persons
edited_count = 0
legs_changed_total = 0
persons_with_car_legs_changed = 0

for person in root.findall("person"):
    pid = person.get("id")
    if pid not in selected_ids:
        continue

    changed = change_car_legs_to_micro_in_selected_plan(person, MICRO_MODE)
    legs_changed_total += changed
    if changed > 0:
        set_person_attr(person, "subpopulation", "potMCUser", java_class="java.lang.String")
        persons_with_car_legs_changed += 1
        edited_count += 1
    # If no leg changed, keep subpopulation unchanged (safety net)

# --- Re-count after edits
total_persons_after, subpops_after = subpopulation_counts(root)

# --- Write output (gzip-aware) with preserved DTD
xml_bytes = etree.tostring(
    tree,
    pretty_print=True,
    xml_declaration=True,
    encoding="UTF-8",
    doctype=DTD_POPULATION
)
write_xml_maybe_gz(OUT_PATH, xml_bytes)

# --- Reporting
print("=== MATSim subpopulation edit summary (probabilistic micro_car conversion) ===")
print(f"Input file:  {IN_PATH}")
print(f"Output file: {OUT_PATH}\n")

print("Eligibility criteria: household_size <= 2, ≥1 car leg in SELECTED plan, and")
print(f"  total_km(car legs) < {DIST_THRESHOLD_KM} km (distance from <leg mode='car'><route distance='...'> in meters)\n")

print(f"Total persons (before): {total_persons_before}")
for sp, c in subpops_before.items():
    print(f"  - {sp:>18}: {c}")

print(f"\nEligible 'person' count: {eligible_scanned} (of {orig_person_count} in 'person')")
print(f"Conversion share requested: {CONVERT_SHARE:.4f} ({CONVERT_SHARE*100:.2f}%)")
print(f"Sampled for conversion:    {len(selected_ids)} (seed={RANDOM_SEED})")

print(f"\nEdited persons (person -> potMCUser, with ≥1 car leg changed): {edited_count}")
print(f"  Share of original 'person' changed: {edited_count} / {orig_person_count} ({pct(edited_count, orig_person_count):.2f}%)")
print(f"  Share relative to total pop:        {edited_count} / {total_persons_before} ({pct(edited_count, total_persons_before):.2f}%)")

print(f"\nLeg mode changes within SELECTED plans:")
print(f"  Persons with ≥1 'car' leg changed: {persons_with_car_legs_changed} / {len(selected_ids) if selected_ids else 1} "
      f"({pct(persons_with_car_legs_changed, len(selected_ids) if selected_ids else 1):.2f}%)")
print(f"  Total legs changed (car -> {MICRO_MODE}): {legs_changed_total}")

print("\nTotal persons (after):", total_persons_after)
for sp, c in subpops_after.items():
    print(f"  - {sp:>18}: {c}")
